# **Day-03 - Spatial Transcriptomics Analysis - Bioinformatics Workshop 2026**

## Loading the library and other data
Download the following files from the UVA-box and upload here


1.   R_library.tar.gz
2.   sparanger_output.tar
3.   weighted_networks_nsga2r_final.rds
4.   lr_network_human_21122021.rds
5.   ligand_target_matrix_nsga2r_final.rds


In [ ]:
#untar("R_library.tar.gz")
untar("library_23July2026.tar.gz")
#list.files("library")

In [ ]:
#untar("R_library.tar.gz")
untar("spaceranger_output.tar")
#list.files("library")

In [ ]:
# change the R library directory into './library'
.libPaths("library")

In [ ]:

library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)


In [ ]:
#install.packages("hdf5r")

This notebook picks up where Space Ranger left off and runs the full spatial analysis pipeline we covered this week: loading data, QC, clustering, cell type annotation (marker-based and reference-based), differential expression between spatial domains, trajectory inference, and ligand-receptor cell-cell communication — all adapted for spatial (not dissociated single-cell) data.

##**Dataset:** Human Prostate Cancer, Adenocarcinoma with Invasive Carcinoma (FFPE) — 10x Genomics, Visium, CC BY 4.0.

**How to use this notebook in the working session:** every code chunk is preceded by a short note on *why* we're doing that step and what decision it feeds into, and followed by a note on what to actually look at in the output before moving on. Nothing here is meant to be run blindly — pause at each decision point (there are several marked clearly below) and look at the plot before continuing.


## 1. Load the Space Ranger Output into Seurat

**Why this step:** `Load10X_Spatial()` is doing more than reading in a matrix — it's stitching together three separate things Space Ranger produced (the filtered feature-barcode matrix, `tissue_positions.csv`, and the tissue image) into one Seurat object. This is the step where "genes × spots" and "where each spot physically sits on the tissue" become linked. Everything downstream — every plot, every cluster, every ligand — depends on this join having worked correctly.

In [ ]:
prostate <- Load10X_Spatial(
  data.dir = "./spaceranger_output/",
  filename = "filtered_feature_bc_matrix.h5",
  assay = "Spatial",
  slice = "prostate"
)

prostate



**What to check:** the printed object should show one `Spatial` assay and a reasonable number of spots (recall from the dataset page: 4,371 spots were called under tissue) and features (~tens of thousands of genes, since this is whole-transcriptome-panel FFPE). If the spot count looks wildly different, something went wrong upstream in the Space Ranger run, not here.

In [ ]:
# Confirm the object actually carries spatial coordinates and the tissue image,
# not just an expression matrix
Assays(prostate)
Images(prostate)
GetTissueCoordinates(prostate) |> head()

**Why we check this explicitly:** it's entirely possible to load a matrix successfully while the image or coordinates silently fail to attach (wrong path, mismatched spot barcodes, etc.). `Images(prostate)` should return the `"prostate"` slice we named above, not `character(0)`. This is a 10-second check that saves an hour of confusion later when `SpatialFeaturePlot()` produces a blank or empty tissue outline.


## 2. QC & Filtering

**Why this step:** spot-level QC in spatial data asks a similar question to cell-level QC in scRNA-seq — "is this spot's data trustworthy?" — but for a different reason. A low-count spot in Visium usually doesn't mean a dying cell (as it would in droplet scRNA-seq); it more often means the spot sits over very little tissue, over a fold or tear in the section, or over an acellular region (like a lumen or a gap in the tissue). We still filter it out, just for a spatial-specific reason.

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 100)
prostate[["percent.mt"]] <- PercentageFeatureSet(prostate, pattern = "^MT-")

VlnPlot(prostate, features = c("nCount_Spatial", "nFeature_Spatial", "percent.mt"),
        pt.size = 0.1, ncol = 3)  & theme(plot.title = element_text(size = 40, face = "bold")) & theme(text=element_text(size=25))

**What to look at:** the same three metrics as single-cell QC (UMI counts, gene counts, percent mitochondrial), but interpreted spatially. FFPE samples in particular often run a bit higher on percent.mt than fresh-frozen tissue due to RNA degradation during fixation — don't reflexively apply a fresh-frozen threshold here.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 18, repr.plot.res = 100)
# Total UMI counts overlaid on the actual tissue image — confirms spatial
# coordinates and image registered correctly during the Section 1 load
SpatialFeaturePlot(prostate, features = "nCount_Spatial") +
  theme(legend.position = "right") +  theme(text=element_text(size=25))


**Why this plot specifically, and why now:** this is the single most useful spatial QC plot, and it does double duty. First, it's a second, more intuitive confirmation that Section 1 loaded correctly — you should be able to visually make out the tissue's shape from the UMI counts alone. Second, low-count spots often aren't randomly scattered — they cluster at the tissue edges, over tears, or over histologically acellular regions (e.g., a duct lumen). Seeing *where* the low-quality spots are tells you something biological/technical that a violin plot alone cannot.

In [ ]:
# Standard spot-level filtering — adjust thresholds after looking at the
# violin plots above; these are reasonable Visium FFPE starting points
prostate <- subset(
  prostate,
  subset = nCount_Spatial >= 500 & nFeature_Spatial >= 200 & percent.mt < 25
)

prostate


**Decision point:** these thresholds are starting points, not fixed rules — set them by eye from the violin plots above, the same way we did for scRNA-seq. Filtering too aggressively in Visium has a cost single-cell filtering doesn't: you're not just discarding a cell, you're creating a literal hole in your tissue map where that spot used to be. Err slightly more permissive than you would for single-cell data.



## 3. Normalization

**Why SCTransform specifically:** we could log-normalize with a single global scale factor, but Visium spots vary a lot in how much tissue and how many cells they actually cover — a spot over dense tumor and a spot over sparse stroma are not directly comparable the way two similarly-sized cells are. `SCTransform()`'s per-gene regression against sequencing depth handles this more explicitly, which is why it's the current default recommendation for spatial data specifically (not just a carryover habit from single-cell).

In [ ]:
prostate <- SCTransform(
  prostate, assay = "Spatial", method = "glmGamPoi", vst.flavor = "v2",
  variable.features.n = 1000,
  n_genes = 1500,     # was 2000 — this is the number in your log, directly controls this step's cost
  ncells = 1500,       # forces a real subsample instead of fitting on all 4306 spots
  return.only.var.genes = TRUE,
  verbose = TRUE
)


**What just happened:** this single call replaces the old three-step `NormalizeData()` + `FindVariableFeatures()` + `ScaleData()` pipeline and adds a new `SCT` assay alongside the original `Spatial` assay — the raw counts in `Spatial` are untouched and still there if you need them later (e.g., for pseudobulk-style aggregation).



## 4. Dimensionality Reduction & Spatial Clustering

**Why this step, and an important caveat up front:** PCA, neighbor-finding, and clustering here are running on expression alone — exactly like single-cell clustering, and *not* using the spatial coordinates at all yet. That's intentional for this first pass, but it means a "cluster" at this stage is a transcriptional grouping that may or may not turn out to respect tissue geography. The next plot is how we check that.

In [ ]:
prostate <- RunPCA(prostate, assay = "SCT", verbose = FALSE)
prostate <- FindNeighbors(prostate, reduction = "pca", dims = 1:30)
prostate <- FindClusters(prostate, verbose = FALSE, resolution = 0.4)
prostate <- RunUMAP(prostate, reduction = "pca", dims = 1:30)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 18, repr.plot.res = 100)
p1 <- DimPlot(prostate, reduction = "umap", label = TRUE, label.size = 8, label.box = TRUE, label.color = "white") + ggtitle("UMAP")  &  theme(text=element_text(size=25))  & theme(legend.position = 'bottom') & guides(colour = guide_legend(override.aes = list(size = 10)))
p1.dim <- SpatialDimPlot(prostate, label = TRUE, label.size = 8) + ggtitle("Spatial") + guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p1 + p1.dim

**This is the key spatial sanity check, worth pausing on as a group:** clusters that are transcriptionally distinct in the UMAP on the left should also show up as spatially coherent regions in the tissue plot on the right — a "malignant epithelium" cluster should look like a contiguous patch on the tissue, not a random sprinkle of spots. A cluster that's tight in UMAP space but scattered all over the tissue image is worth a second look: it could be a real, spatially-distributed cell type (e.g., a diffuse immune infiltrate), or it could be a batch/technical artifact. The whole point of spatial data is that we get to ask this question at all — single-cell data has no equivalent check.

> **Alternative worth mentioning here:** everything above clusters on expression only. Tools like **BayesSpace** or **STutility** build spatial coordinates directly into the clustering step itself (smoothing cluster assignments toward spatial neighbors), which can produce cleaner spatial domains. We're using the expression-only approach here because it's what ships in Seurat directly, but it's worth trying the alternative on your own data if domains look noisy.



## 5. Cell Type / Domain Annotation

**Why annotation looks different here than in the scRNA-seq lecture:** a Visium spot is ~55µm across and can contain anywhere from one to ten-plus cells. So we're not assigning "this spot is a macrophage" the way we'd label a single cell — we're asking "what mixture of cell types and tissue structures dominates this spot," which is a softer, more compositional question. Keep that framing in mind through this whole section.

### 5a. Marker-based scoring

**Why start here, and why these specific genes:** rather than jumping straight to an automated tool, we score each spot directly against genes we already know are diagnostic for prostate tissue — the same "look at markers you trust" instinct from the single-cell lecture, just computed per spot instead of per cell.

**Ground truth for this section:** the pathology report for this section identifies seven tissue categories present: **Invasive Carcinoma, Normal Gland, Fibro-Muscular Tissue, Fibrous Tissue, Immune Cells, Blood Vessel,** and **Nerve**. The marker panels below are built to match these exact categories, so the computational calls in this notebook can be compared directly against a real pathologist's read of the same tissue — a genuinely rare and valuable validation opportunity that most workshop datasets don't have.

**One important biological detail baked into the "Normal Gland" panel:** the histological feature that actually distinguishes invasive carcinoma from a normal gland is the loss of the **basal cell layer** — normal prostate glands have an intact outer layer of basal cells surrounding the luminal cells; invasive carcinoma has lost this layer entirely. That's exactly why TP63/KRT5/KRT14 (basal markers) are grouped with the luminal markers under "Normal Gland" here, rather than split out separately — their presence *is* the definition of "still a normal gland," and their absence alongside AMACR/ERG positivity is what "invasive" means.

In [ ]:
options(repr.plot.width = 40, repr.plot.height = 35, repr.plot.res = 100)
# Marker panels matched to the pathologist's 7 annotated tissue categories
invasive_carcinoma <- c("AMACR", "ERG")
normal_gland        <- c("KLK3", "KLK2", "NKX3-1", "AR", "KRT5", "KRT14", "TP63")
fibromuscular       <- c("ACTA2", "MYH11", "DES")
fibrous_tissue      <- c("DCN", "LUM", "COL1A1", "COL1A2")
immune_cells        <- c("PTPRC", "CD3E", "CD68", "MS4A1")
blood_vessel        <- c("PECAM1", "VWF")
nerve               <- c("S100B", "SOX10", "NGFR", "MPZ")

prostate <- AddModuleScore(prostate, features = list(invasive_carcinoma), name = "InvasiveCarcinoma")
prostate <- AddModuleScore(prostate, features = list(normal_gland), name = "NormalGland")
prostate <- AddModuleScore(prostate, features = list(fibromuscular), name = "FibroMuscular")
prostate <- AddModuleScore(prostate, features = list(fibrous_tissue), name = "FibrousTissue")
prostate <- AddModuleScore(prostate, features = list(immune_cells), name = "ImmuneCells")
prostate <- AddModuleScore(prostate, features = list(blood_vessel), name = "BloodVessel")
prostate <- AddModuleScore(prostate, features = list(nerve), name = "Nerve")

SpatialFeaturePlot(prostate, features = c("InvasiveCarcinoma1", "NormalGland1", "FibrousTissue1", "FibroMuscular1", "ImmuneCells1", "BloodVessel1", "Nerve1")) &  theme(text=element_text(size=15))


**Why a module score instead of one gene at a time:** any single marker gene is noisy at the per-spot level (dropout affects spatial data too, just like single-cell). Averaging several genes into one score (`AddModuleScore()`) smooths out that noise and gives a more stable per-spot signal for "how much does this spot look like program X."

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 20, repr.plot.res = 100)

# Individual markers are often more interpretable than module scores for a
# first look — KLK3 (PSA) and AMACR are the two most directly diagnostic here
SpatialFeaturePlot(prostate, features = c("KLK3", "AMACR", "TP63", "EPCAM")) &  theme(text=element_text(size=15))


**Why also look at raw genes, not just scores:** module scores are great for orientation but can obscure which specific gene is actually driving a pattern. Looking at KLK3, AMACR, and EPCAM individually here is the direct, defensible, "what would a pathologist ask for" version of the same question.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
# Which clusters actually score highest for each program? This is the bridge
# from "marker gene analysis" to "naming cell types" from the annotation lecture
VlnPlot(prostate, features = c("InvasiveCarcinoma1", "NormalGland1", "FibrousTissue1", "ImmuneCells1"),
        pt.size = 0, ncol = 2) &  theme(text = element_text(size = 24, face="bold")) + theme(axis.text=element_text(size=18))

**Why this is the bridge step:** the spatial feature plots tell you where each program is active; this violin plot tells you which *cluster number* from Section 4 that corresponds to. That mapping — cluster 2 scores high for `Stromal1`, so cluster 2 is "stroma" — is exactly what `FindAllMarkers()` below will help you confirm and refine.

In [ ]:
cluster_markers <- FindAllMarkers(prostate, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
cluster_markers %>% group_by(cluster) %>% slice_max(avg_log2FC, n = 5)

**Why run this even though we already have marker scores:** the module scores above only tell you about the genes we already thought to check. `FindAllMarkers()` asks the more open-ended question — "what genes actually distinguish this cluster from every other cluster" — and will often surface something we didn't put in a panel, including possibly a signature specific to this individual tumor rather than prostate tissue in general.

In [ ]:
cluster_markers %>%
  group_by(cluster) %>%
  dplyr::filter(avg_log2FC > 1) %>%
  slice_head(n = 3) %>%
  ungroup() -> top3

In [ ]:
cluster_markers %>%
  group_by(cluster) %>%
  dplyr::filter(avg_log2FC > 1) %>%
  slice_head(n = 5) %>%
  ungroup() -> top5

In [ ]:
prostate <- ScaleData(prostate,features = top3$gene)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
p <- DoHeatmap(prostate, features = top3$gene, size = 6) + theme(axis.text = element_text(size = 15))   & theme(axis.text.y = element_text(size = 24, face="bold"))  + theme(text=element_text(size=25, face="bold"))
p

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
SpatialFeaturePlot(prostate, features=c(filter(top5, cluster == 0)$gene), alpha=c(0.1,1.5), pt.size.factor = 2.5, image.alpha=0.2) &  theme(text=element_text(size=15))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 100)
VlnPlot(prostate, features = c(filter(top5, cluster == 0)$gene), layer = 'data', split.by = "orig.ident", split.plot = F, stack = T, flip = T, pt.size = 0.01) + theme(axis.text=element_text(size=18))

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
SpatialFeaturePlot(prostate, features=c(filter(top5, cluster == 7)$gene), alpha=c(0.1,1.5), pt.size.factor = 2.5, image.alpha=0.2) &  theme(text=element_text(size=15))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 100)
VlnPlot(prostate, features = c(filter(top5, cluster == 7)$gene), layer = 'data', split.by = "orig.ident", split.plot = F, stack = T, flip = T, pt.size = 0.01) + theme(axis.text=element_text(size=18))

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
SpatialFeaturePlot(prostate, features=c(filter(top5, cluster == 6)$gene),alpha=c(0.1,1.5),pt.size.factor = 2.5,image.alpha=0.2) &  theme(text=element_text(size=15))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 100)
VlnPlot(prostate, features = c(filter(top5, cluster == 6)$gene), layer = 'data', split.by = "orig.ident", split.plot = F, flip = T, pt.size = 0.01) + theme(axis.text=element_text(size=18))


**Why run this even though we already have marker scores:** the module scores above only tell you about the genes we already thought to check. `FindAllMarkers()` asks the more open-ended question — "what genes actually distinguish this cluster from every other cluster" — and will often surface something we didn't put in a panel, including possibly a signature specific to this individual tumor rather than prostate tissue in general.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
# Turn the continuous module scores into one discrete label per spot —
# the automated, ScType-style version of marker-based annotation from the
# single-cell lecture, just running on spots instead of cells
program_scores <- FetchData(prostate, vars = c(
  "InvasiveCarcinoma1", "NormalGland1", "FibroMuscular1", "FibrousTissue1",
  "ImmuneCells1", "BloodVessel1", "Nerve1"
))

# Whichever program scores highest for a spot becomes its marker-based call
prostate$marker_call <- colnames(program_scores)[max.col(program_scores, ties.method = "first")]
prostate$marker_call <- gsub("1$", "", prostate$marker_call)  # tidy the AddModuleScore suffix

p2.dim.module <- SpatialDimPlot(prostate, group.by = "marker_call", label = TRUE, label.size = 8) + ggtitle("Marker-based call (per spot)") + guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p2.dim.module


**Why this discrete step matters, not just the continuous scores above:** the module scores tell you *how much* each program is active, but don't commit to an answer — this is the step that mirrors what an automated marker-based tool like ScType actually outputs: one label per unit (spot here, cell in the single-cell lecture), assigned by whichever program wins. It's a genuinely useful first-pass label, and also a useful sanity check against the cluster-based naming from `FindAllMarkers()` above — where the two disagree is worth a closer look, exactly like the "Azimuth vs. markers disagree" discussion from the single-cell lecture. Since we have the pathologist's own categories to compare against here, this is also the step to eyeball directly against the pathology report if a region-level annotation image or file is available — that's the closest thing to real ground truth a workshop dataset ever gets.


In [ ]:
# EDIT THIS after reviewing the marker scores and FindAllMarkers output above —
# cluster numbers are data-dependent and will not match this template exactly.
# using the pathologist's own category names:
 new_labels <- c(
   "0" = "Invasive Carcinoma",
   "1" = "Fibro-Muscular Tissue",
   "2" = "Normal Gland",
   "3" = "Invasive Carcinoma",
   "4" = "Normal Gland",
   "5" = "Immune Cells",
   "6" = "Fibrous Tissue",
   "7" = "Normal Gland",
   "8" = "Nerve"
 )
 prostate <- RenameIdents(prostate, new_labels)
 prostate$domain <- Idents(prostate)
prostate@meta.data$manual_labels <- prostate@active.ident

**Decision point — this is a group discussion, not a formula:** naming clusters is a judgment call informed by the module scores, the marker-based calls, the raw marker genes, and the `FindAllMarkers()` output together — no single number here tells you definitively "this is the malignant cluster." This is the same annotation judgment call from the lecture, just now informed by *where* the cluster sits on the tissue as an extra line of evidence single-cell annotation never gets — and, uniquely for this dataset, informed by an actual pathologist's read of the same seven categories to check your work against.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
#SpatialDimPlot(prostate, group.by = "new_labels") + ggtitle("Marker-based call (per spot)")
p3.dim.manual <- SpatialDimPlot(prostate, label = TRUE, label.size = 8) + ggtitle("Spatial") & guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p3.dim.manual

In [ ]:
saveRDS(object = prostate, file = "spatial_prostate.rds")

### Automated marker-based annotation (ScType)

**Why also run the actual published tool, not just our hand-rolled version above:** the discrete marker-based call above is a simplified version of exactly what **ScType** (Ianevski et al., *Nat Commun* 2022) does formally — it's a widely used automated marker-based annotation tool that scores every cell (or spot) against a database of cell-type marker panels and assigns whichever type scores highest, with proper handling of both positive markers (genes a type *should* express) and negative markers (genes it *shouldn't*), which our simple `max.col()` approach above ignores entirely.

**Why we can't just use ScType's built-in database as-is:** ScType ships with marker panels for specific tissues (Immune system, Pancreas, Liver, Eye, Kidney, Brain, Lung, Adrenal, Heart, Intestine, Muscle, Placenta, Spleen, Stomach, Thymus) — and prostate isn't one of them. Rather than treat that as a dead end, ScType is explicitly designed to accept a custom marker database in the same format, which means we can plug in exactly the prostate marker panels we already defined in Section 5a.

---



In [ ]:
#install.packages("HGNChelper")
#install.packages("openxlsx")

In [ ]:
library(HGNChelper)
library(openxlsx)

# ScType's scoring functions (not a CRAN package — sourced directly, as the
# tool's own documentation instructs)
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R")
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")

# Build a custom marker database in ScType's required 4-column format, matched
# to the pathologist's own 7 tissue categories from Section 5a
custom_markers <- data.frame(
  tissueType = "Prostate",
  cellName = c("Invasive Carcinoma", "Normal Gland", "Fibro-Muscular Tissue",
               "Fibrous Tissue", "Immune Cells", "Blood Vessel", "Nerve"),
  geneSymbolmore1 = c(
    "AMACR,ERG",
    "KLK3,KLK2,NKX3-1,AR,KRT5,KRT14,TP63",
    "ACTA2,MYH11,DES",
    "DCN,LUM,COL1A1,COL1A2",
    "PTPRC,CD3E,CD68,MS4A1",
    "PECAM1,VWF",
    "S100B,SOX10,NGFR,MPZ"
  ),
  geneSymbolmore2 = c(
    "KRT5,KRT14,TP63",  # invasive carcinoma should NOT retain the basal layer
    "", "", "", "", "", ""
  )
)

marker_file <- tempfile(fileext = ".xlsx")
openxlsx::write.xlsx(custom_markers, marker_file)

gs_list <- gene_sets_prepare(marker_file, "Prostate")

In [ ]:
all_marker_genes <- unique(unlist(strsplit(
  paste(c(custom_markers$geneSymbolmore1, custom_markers$geneSymbolmore2), collapse = ","), ","
)))
all_marker_genes <- all_marker_genes[all_marker_genes != ""]

# Make sure every gene ScType needs actually exists in scale.data
prostate <- ScaleData(prostate, assay = "SCT",
                       features = union(VariableFeatures(prostate), all_marker_genes))

scaled_data <- as.matrix(prostate[["SCT"]]$scale.data)

In [ ]:
# ScType scores against the SCT scale.data matrix, same input PCA/clustering used
scaled_data <- as.matrix(prostate[["SCT"]]$scale.data)

es.max <- sctype_score(scRNAseqData = scaled_data, scaled = TRUE,
                        gs = gs_list$gs_positive, gs2 = gs_list$gs_negative)

# ScType's standard aggregation: sum per-spot scores within each cluster,
# then take the top-scoring cell type for that cluster as a whole
cL_results <- do.call("rbind", lapply(unique(prostate$seurat_clusters), function(cl) {
  spots_in_cluster <- rownames(prostate@meta.data[prostate$seurat_clusters == cl, ])
  es.max.cl <- sort(rowSums(es.max[, spots_in_cluster, drop = FALSE]), decreasing = TRUE)
  head(data.frame(cluster = cl, type = names(es.max.cl), score = es.max.cl), 1)
})) %>% arrange(cluster)

cL_results

In [ ]:
cluster_to_sctype <- setNames(cL_results$type, cL_results$cluster)
#prostate$sctype_label <- cluster_to_sctype[as.character(prostate$seurat_clusters)]

#SpatialDimPlot(prostate, group.by = "sctype_label") + ggtitle("ScType call (per cluster)")
cluster_to_sctype

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
sctype_vec <- cluster_to_sctype[as.character(prostate$seurat_clusters)]
names(sctype_vec) <- NULL   # <- this line fixes it

prostate$sctype_label <- sctype_vec

p4.dim.scType <- SpatialDimPlot(prostate, group.by = "sctype_label", label = TRUE, label.size = 8) + ggtitle("ScType call (per cluster)") + guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p4.dim.scType

## Display all annotated plots together

In [ ]:
options(repr.plot.width = 50, repr.plot.height = 15, repr.plot.res = 100)
p1.dim + p2.dim.module + p3.dim.manual + p4.dim.scType +
  plot_layout(nrow = 1)

In [ ]:
# 1. Master color palette — one entry per possible label, defined once
celltype_colors <- c(
  "InvasiveCarcinoma" = "#E41A1C",
  "NormalGland"       = "#4DAF4A",
  "FibroMuscular"      = "#984EA3",
  "FibrousTissue"      = "#FF7F00",
  "ImmuneCells"        = "#377EB8",
  "BloodVessel"        = "#F781BF",
  "Nerve"              = "green"
)
celltype_colors1 <- c(
  "Invasive Carcinoma" = "#E41A1C",
  "Normal Gland"       = "#4DAF4A",
  "Fibro-Muscular Tissue"      = "#984EA3",
  "Fibrous Tissue"      = "#FF7F00",
  "Immune Cells"        = "#377EB8",
  "Blood Vessel"        = "#F781BF",
  "Nerve"              = "green"
)

# 2. Make sure each label column is a factor with the SAME levels/order —
#    this matters even more than the colors themselves
prostate$manual_labels <- factor(prostate$manual_labels,  levels = names(celltype_colors1))
prostate$marker_call   <- factor(prostate$marker_call,   levels = names(celltype_colors))
prostate$sctype_label  <- factor(prostate$sctype_label,  levels = names(celltype_colors1))


# 3. Pass the same named vector to every plot
p2.dim.manual <- SpatialDimPlot(prostate, group.by = "manual_labels", cols = celltype_colors1, label = TRUE, label.size = 8) & guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p3.dim.module <- SpatialDimPlot(prostate, group.by = "marker_call",  cols = celltype_colors, label = TRUE, label.size = 8) &  guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')
p4.dim.scType <- SpatialDimPlot(prostate, group.by = "sctype_label", cols = celltype_colors1, label = TRUE, label.size = 8) &  guides(colour = guide_legend(override.aes = list(size = 10))) &  theme(text=element_text(size=25))   & theme(legend.position = 'bottom')

(p1.dim + p2.dim.manual + p3.dim.module + p4.dim.scType) +
  plot_layout(nrow = 1, guides = "collect")


## Differential Expression Between Spatial Domains

**Why the comparison is different from the scRNA-seq lecture:** the lecture's DE section compared pre- vs. post-treatment samples — two different tissue sections, matched by patient. We only have one tissue section here, so that comparison isn't available. The natural spatial equivalent is comparing **regions within the same section**: malignant vs. benign epithelium, or tumor vs. adjacent stroma. It's a different question (spatial heterogeneity within one tumor, rather than treatment response across two timepoints) but uses identical mechanics.

**Why we're running this two ways:** the lecture's central DE lesson was pseudobulk vs. spot/cell-level testing, and this is the moment to actually see that difference on real data rather than just hear about it. We'll run both `FindMarkers()` (spot-level) and a pseudobulk-style DESeq2 comparison on the *same two clusters*, then look at both volcano plots side by side.

In [ ]:
# Replace with your actual assigned domain labels from Section 5a once you
# have them — cluster numbers are data-dependent
Idents(prostate) <- "seurat_clusters"
cluster_A <- "0"   # EDIT: e.g. Invasive Carcinoma
cluster_B <- "3"   # EDIT: e.g. Normal Gland

### Spot-level DE (`FindMarkers`)

**Why start here:** this is the same function and the same call pattern from the single-cell lecture — one cluster's spots vs. another's, treating every spot as an independent observation. It's fast, and it's exactly what most people reach for first. Keep the lecture's caveat in mind while it runs: every spot here is being treated as a replicate, and neighboring spots on the same tissue are not independent measurements the way separate patient samples are.

In [ ]:
de_spotlevel <- FindMarkers(
  prostate,
  ident.1 = cluster_A,
  ident.2 = cluster_B,
  assay = "SCT",
  test.use = "wilcox",
  min.pct = 0.1,
  logfc.threshold = 0.1
)

de_spotlevel <- de_spotlevel %>% arrange(p_val_adj)
head(de_spotlevel, 20)

**What to notice here, before looking at the pseudobulk version:** count roughly how many points fall in the red "significant" zone, and note how tight the p-values get (points near the top of the plot). Hold that number in your head for the comparison below.


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15, repr.plot.res = 100)
library(ggrepel)

# --- Spot-level volcano, labeled ---
de_spotlevel$sig <- with(de_spotlevel,
  ifelse(p_val_adj < 0.05 & abs(avg_log2FC) > 1, "Significant", "NS"))

top_genes_spotlevel <- de_spotlevel %>%
  filter(sig == "Significant") %>%
  arrange(p_val_adj) %>%
  slice_head(n = 15) %>%
  tibble::rownames_to_column("gene")

volcano_spotlevel <- ggplot(de_spotlevel, aes(avg_log2FC, -log10(p_val_adj), color = sig)) +
  geom_point(alpha = 0.5, size = 3) +
  scale_color_manual(values = c(NS = "grey70", Significant = "red")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
  geom_text_repel(
    data = top_genes_spotlevel,
    aes(avg_log2FC, -log10(p_val_adj), label = gene),
    color = "black", size = 6, max.overlaps = 20,
    box.padding = 0.4, segment.size = 0.3
  ) +
  theme_minimal() +
  labs(title = "Spot-level (FindMarkers)") + theme(text=element_text(size=25))

volcano_spotlevel

## Trajectory Analysis (Monocle3)

**Why a trajectory analysis makes sense for *this specific tissue* — pause and discuss:** this is exactly the judgment call from the "trajectory or not?" discussion in the lecture. This tissue is annotated as adenocarcinoma **with invasive carcinoma**, which suggests it plausibly contains a real biological continuum — benign epithelium grading into increasingly malignant, invasive tissue — rather than a set of unrelated, discrete cell types. That continuum is what makes pseudotime a sensible tool to reach for here. If your own tissue is more like "three unrelated cell types sitting next to each other," this section is the wrong tool, no matter how easily the code runs.

In [ ]:
#devtools::install_github('cole-trapnell-lab/monocle3')

In [ ]:
#install.packages("R.utils")


In [ ]:
library(R.utils)
library(monocle3)

In [ ]:
seurat_to_cds <- function(seurat_obj, assay = "SCT") {
  expr_matrix   <- GetAssayData(seurat_obj, assay = assay, layer = "counts")
  cell_metadata <- seurat_obj@meta.data
  gene_metadata <- data.frame(gene_short_name = rownames(expr_matrix),
                               row.names = rownames(expr_matrix))

  cds <- new_cell_data_set(
    expression_data = expr_matrix,
    cell_metadata   = cell_metadata,
    gene_metadata   = gene_metadata
  )

  reducedDims(cds)$PCA  <- Embeddings(seurat_obj, "pca")
  reducedDims(cds)$UMAP <- Embeddings(seurat_obj, "umap")

  cds
}

cds <- seurat_to_cds(prostate, assay = "SCT")
cds <- cluster_cells(cds)
cds <- learn_graph(cds)

**What's happening mechanically:** `as.cell_data_set()` converts our Seurat object into Monocle3's format (carrying over the SCT-normalized expression and existing UMAP), `cluster_cells()` re-clusters within Monocle3's own framework (it keeps its own cluster assignments separate from Seurat's), and `learn_graph()` fits the principal graph — the tree-like structure pseudotime will be measured along.

In [ ]:
head(prostate@meta.data)

In [ ]:
# Pick a root: the benign epithelium cluster identified in Section 5a
# (interactive selection is easiest in a live session)
root_cells <- colnames(prostate)[prostate$manual_labels == "Normal Gland"]

cds <- order_cells(cds, root_cells = root_cells)

**Why the root choice matters and isn't automatic:** pseudotime is a *relative* measure of distance from a starting point you choose — Monocle3 has no biological knowledge of which end of the graph is "the beginning." Rooting at the benign epithelium cluster is what makes the resulting pseudotime interpretable as "progression toward malignancy" rather than an arbitrary number. This is a deliberate biological choice, not a default the software picks for you.

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 15, repr.plot.res = 100)
p1.monocle <- plot_cells(cds, color_cells_by = "pseudotime", label_cell_groups = FALSE,
           label_leaves = FALSE, label_branch_points = FALSE, cell_size = 2) + theme(text=element_text(size=25))
p1.monocle

In [ ]:
p1.clus <- DimPlot(prostate, reduction = "umap", label = TRUE, group.by = 'manual_labels') + ggtitle("UMAP")

In [ ]:
options(repr.plot.width = 35, repr.plot.height = 15, repr.plot.res = 100)
p1.clus + p1.monocle

# Bring pseudotime back into Seurat and view it on the actual tissue —
# the step that makes spatial trajectory analysis distinct from single-cell
prostate$pseudotime <- pseudotime(cds)
SpatialFeaturePlot(prostate, features = "pseudotime")

In [ ]:
# Bring pseudotime back into Seurat and view it on the actual tissue —
# the step that makes spatial trajectory analysis distinct from single-cell
options(repr.plot.width = 25, repr.plot.height = 15, repr.plot.res = 100)
prostate$pseudotime <- pseudotime(cds)
p2.pseudotime <- SpatialFeaturePlot(prostate, features = "pseudotime")  + theme(text=element_text(size=25))
p2.pseudotime

In [ ]:
options(repr.plot.width = 40, repr.plot.height = 15, repr.plot.res = 100)
(p1.dim + p2.dim.manual + p2.pseudotime) +
  plot_layout(nrow = 1, guides = "collect")


**Why this last plot is the whole point of doing trajectory analysis on spatial data specifically:** in single-cell data, pseudotime only ever lives in an abstract UMAP/graph space. Here, we can put it back on the actual tissue image and ask a question single-cell data can't answer: does the transcriptional progression Monocle3 found actually correspond to a spatial progression — e.g., does pseudotime increase as you move from the tissue's benign region toward the invasive front? If it does, that's strong independent evidence the trajectory is real biology, not a computational artifact.



**Reading the actual result honestly:** running this on all clusters together (including Immune Cells, Nerve, Fibro-Muscular Tissue — populations with no plausible role in an epithelial benign-to-malignant continuum) tends to produce a messier picture than the clean story above: a small, high-pseudotime "hot zone" right at the Invasive Carcinoma/Normal Gland border, most of Invasive Carcinoma sitting at a middling value rather than uniformly high, and every unrelated population reading near zero simply because they're far from the root in expression space — not because they're biologically "early." That's not a bug; it's the graph honestly reflecting that unrelated cell types were forced onto one connected trajectory. Which is exactly why the next step restricts the trajectory to only the populations that plausibly belong on it.


### 7a. A cleaner test: restrict the trajectory to epithelium only

**Why redo this on a subset:** the whole premise of pseudotime is a continuum connecting biologically related states. Immune cells, nerve, and stroma aren't points along a benign-to-malignant epithelial continuum — they're different cell types entirely, just physically nearby. Including them doesn't just add noise, it can actively distort the graph Monocle3 fits. Restricting to Normal Gland + Invasive Carcinoma tests the actual hypothesis of interest directly.

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 15, repr.plot.res = 100)
epithelial_cds <- cds[, colData(cds)$manual_labels %in% c("Normal Gland", "Invasive Carcinoma")]

epithelial_cds <- cluster_cells(epithelial_cds)
epithelial_cds <- learn_graph(epithelial_cds)

root_cells_epi <- colnames(epithelial_cds)[colData(epithelial_cds)$manual_labels == "Normal Gland"]
epithelial_cds <- order_cells(epithelial_cds, root_cells = root_cells_epi)

p3_monocle <- plot_cells(epithelial_cds, color_cells_by = "pseudotime", label_branch_points = FALSE, cell_size = 2) + theme(text=element_text(size=25))
p3_monocle

In [ ]:
options(repr.plot.width = 35, repr.plot.height = 15, repr.plot.res = 100)
p1 + p1.clus + p3_monocle

**What to expect, and what would actually confirm the hypothesis:** a noticeably cleaner, more monotonic gradient from Normal Gland toward Invasive Carcinoma than the whole-tissue version above — fewer branch points, less "everything unrelated reads as zero" noise. If that shows up, put it next to the whole-tissue plot and the spatial pseudotime plot as a three-way comparison: that progression is genuinely useful evidence for a real benign-to-malignant transcriptional continuum in this tumor, not just an artifact of forcing unrelated cell types onto the same graph.


In [ ]:
saveRDS(object = prostate, file = "spatial_prostate_1.rds")

In [ ]:
table(prostate$manual_labels)

## Cell-Cell Communication (NicheNet)

**Why sender/receiver are spatial domains here, not cell types:** in the lecture's HNSCC example, NicheNet asked whether cancer-associated fibroblasts signal to malignant cells. Our sender/receiver populations are the spatial domains from Section 5 — e.g., does the stroma/immune region signal to the adjacent malignant epithelium region? Same underlying question, same tool, just with spatial-domain labels standing in for the dissociated cell-type labels from the lecture's example.

In [ ]:
#devtools::install_github("saeyslab/nichenetr")

In [ ]:
library(nichenetr)


In [ ]:
library(tidyverse)

In [ ]:

# Prior model download (once; see NicheNet documentation for current links)

#if(organism == "human"){
#  lr_network <- readRDS(url("https://zenodo.org/record/7074291/files/lr_network_human_21122021.rds"))
#  ligand_target_matrix <- readRDS(url("https://zenodo.org/record/7074291/files/ligand_target_matrix_nsga2r_final.rds"))
#  weighted_networks <- readRDS(url("https://zenodo.org/record/7074291/files/weighted_networks_nsga2r_final.rds"))
#} else if(organism == "mouse"){
#  lr_network <- readRDS(url("https://zenodo.org/record/7074291/files/lr_network_mouse_21122021.rds"))
#  ligand_target_matrix <- readRDS(url("https://zenodo.org/record/7074291/files/ligand_target_matrix_nsga2r_final_mouse.rds"))
#  weighted_networks <- readRDS(url("https://zenodo.org/record/7074291/files/weighted_networks_nsga2r_final_mouse.rds"))
#}
organism = "human"

lr_network <- readRDS("./lr_network_human_21122021.rds")
ligand_target_matrix <- readRDS("ligand_target_matrix_nsga2r_final.rds")
weighted_networks <- readRDS("weighted_networks_nsga2r_final.rds")

In [ ]:

Idents(prostate) <- "manual_labels"  # or your assigned domain labels
sender_ids   <- "Invasive Carcinoma"
receiver_ids <- "Normal Gland"
#receiver_ids <- "Fibrous Tissue"

#Idents(prostate) <- "seurat_clusters"  # or your assigned domain labels
#sender_ids   <- "0"
#receiver_ids <- "3"


# Gene set of interest: genes upregulated in the RECEIVER domain specifically,
# relative to the rest of the tissue — computed directly from sender_ids/
# receiver_ids here, rather than reusing Section 6's de_spotlevel (which was
# an independent comparison between different, arbitrarily-chosen clusters)
de_receiver_vs_rest <- FindMarkers(
  prostate, ident.1 = receiver_ids, assay = "SCT",
  test.use = "wilcox", min.pct = 0.1, logfc.threshold = 0.1
)
geneset_oi <- de_receiver_vs_rest %>% filter(p_val_adj < 0.05, avg_log2FC > 0.25) %>% rownames()

# Expressed genes computed ONLY from the sender/receiver spots respectively —
# this is the actual bug fix: these must be subset to sender_ids/receiver_ids,
# not computed from the whole tissue, or the sender/receiver choice never
# affects anything downstream
sender_spots   <- WhichCells(prostate, idents = sender_ids)
receiver_spots <- WhichCells(prostate, idents = receiver_ids)

sct_counts <- GetAssayData(prostate, assay = "SCT", layer = "counts")

expressed_genes_sender   <- rownames(prostate)[rowSums(sct_counts[, sender_spots] > 0) > 10]
expressed_genes_receiver <- rownames(prostate)[rowSums(sct_counts[, receiver_spots] > 0) > 10]

ligands  <- lr_network %>% pull(from) %>% unique()
receptors <- lr_network %>% pull(to) %>% unique()
expressed_ligands   <- intersect(ligands, expressed_genes_sender)
expressed_receptors <- intersect(receptors, expressed_genes_receiver)

potential_ligands <- lr_network %>%
  filter(from %in% expressed_ligands & to %in% expressed_receptors) %>%
  pull(from) %>% unique()

ligand_activities <- predict_ligand_activities(
  geneset = geneset_oi,
  background_expressed_genes = expressed_genes_receiver,
  ligand_target_matrix = ligand_target_matrix,
  potential_ligands = potential_ligands
)

ligand_activities %>% arrange(desc(aupr_corrected)) %>% head(10)

In [ ]:
best_upstream_ligands <- ligand_activities %>% top_n(10, aupr_corrected) %>% pull(test_ligand)

# --- Ligand -> predicted target gene heatmap ---
active_ligand_target_links_df <- best_upstream_ligands %>%
  lapply(get_weighted_ligand_target_links, geneset = geneset_oi,
         ligand_target_matrix = ligand_target_matrix, n = 200) %>%
  bind_rows() %>%
  drop_na()

active_ligand_target_links <- prepare_ligand_target_visualization(
  ligand_target_df = active_ligand_target_links_df,
  ligand_target_matrix = ligand_target_matrix
)

order_ligands <- intersect(best_upstream_ligands, colnames(active_ligand_target_links))
order_targets <- active_ligand_target_links_df$target %>% unique() %>%
  intersect(rownames(active_ligand_target_links))

vis_ligand_target <- active_ligand_target_links[order_targets, order_ligands] %>% t()

options(repr.plot.width = 35, repr.plot.height = 15, repr.plot.res = 100)
ligand_target_heatmap <- make_heatmap_ggplot(
  vis_ligand_target, y_name = "Prioritized ligand", x_name = "Predicted target gene",
  color = "purple", legend_title = "Regulatory potential"
)  + theme(text=element_text(size=20))

ligand_target_heatmap

**How to read this heatmap:** each row is one of the top 10 predicted ligands from the sender domain; each column is a gene from our Section 6 DE gene set (upregulated in the receiver domain). A dark cell means NicheNet's prior model predicts that ligand plausibly drives that specific target gene's expression — this is the direct visual version of "which of the sender's ligands best explain the expression changes I already found" from the lecture. A ligand with mostly light cells across the DE gene set is a weaker candidate despite ranking in the top 10 on `aupr_corrected` alone; look for ligands with several genuinely dark cells, not just a high overall score.

In [ ]:
# --- Ligand -> receptor prior-interaction heatmap ---
lr_network_top_df_large <- get_weighted_ligand_receptor_links(
  best_upstream_ligands, expressed_receptors, lr_network, weighted_networks$lr_sig
)

# NOTE: this object uses lr_network's own "from"/"to" naming (ligand/receptor),
# not "ligand"/"receptor" — check colnames(lr_network_top_df_large) if this
# ever errors after a nichenetr update
lr_network_top_df <- lr_network_top_df_large %>%
  tidyr::spread(from, weight, fill = 0)

lr_network_top_matrix <- lr_network_top_df %>% dplyr::select(-to) %>% as.matrix()
rownames(lr_network_top_matrix) <- lr_network_top_df$to

options(repr.plot.width = 25, repr.plot.height = 15, repr.plot.res = 100)
ligand_receptor_heatmap <- make_heatmap_ggplot(
  lr_network_top_matrix, y_name = "Receptor expressed by receiver", x_name = "Ligand",
  color = "mediumvioletred", legend_title = "Prior interaction potential"
) + theme(text=element_text(size=25))

ligand_receptor_heatmap


**Why this is a genuinely different question from the ligand-target heatmap above:** the ligand-target heatmap asks "does this ligand plausibly explain the *downstream gene expression changes*" — it's mechanistic but indirect. This ligand-receptor heatmap asks the more literal question: "does this ligand have a known receptor that the receiver domain actually expresses?" It's the direct molecular pairing — useful as a sanity check, since a ligand can score well on target prediction while its most obvious receptor is barely expressed in the receiver, which is worth knowing before treating a predicted ligand as a strong candidate.


In [ ]:
options(repr.plot.width = 35, repr.plot.height = 15, repr.plot.res = 100)
# The genuinely spatial add-on: do the top predicted sender ligands actually
# co-localize near the receiver domain, or are sender/receiver on opposite
# ends of the tissue? This is the proximity check dissociated scRNA-seq can't do.
SpatialFeaturePlot(prostate, features = best_upstream_ligands[1:4])  + theme(text=element_text(size=25))



**Why this final plot matters more here than it would for dissociated data:** NicheNet's ranking is still just a prediction based on expression, exactly as discussed in the lecture — high expression of a ligand and its receptor doesn't guarantee real signaling. But because we have spatial coordinates, we can do something the HNSCC scRNA-seq example couldn't: check directly whether the top-ranked ligand is actually expressed physically near the receiver domain, rather than on the opposite side of the tissue. A top-ranked ligand with no spatial proximity to the receiver is a good candidate to be more skeptical of.


In [ ]:
options(repr.plot.width = 40, repr.plot.height = 15, repr.plot.res = 100)
# The single most direct spatial check: plot the top ligand-receptor PAIR
# together, not just the ligand alone, and look for spots where both light up
# NOTE: lr_network_top_df_large uses "from"/"to" (ligand/receptor) naming
top_pair <- lr_network_top_df_large %>% dplyr::arrange(desc(weight)) %>% dplyr::slice(1)
top_ligand   <- top_pair$from
top_receptor <- top_pair$to

p_ligand   <- SpatialFeaturePlot(prostate, features = top_ligand) +
  ggtitle(paste0(top_ligand, " (ligand, sender)"))
p_receptor <- SpatialFeaturePlot(prostate, features = top_receptor) +
  ggtitle(paste0(top_receptor, " (receptor, receiver)"))

p2.dim.manual + p_ligand + p_receptor + plot_layout(nrow = 1)

In [ ]:
top_pair


**What this specific plot is checking:** the single clearest visual test of whether this predicted interaction is spatially plausible at all — the ligand should be visibly concentrated in (or near) the sender domain, and the receptor should be visibly concentrated in the receiver domain, with the two regions actually adjacent to each other on the tissue. If the ligand lights up on one side of the section and the receptor lights up on the far opposite side with no shared border between them, that's a real reason for skepticism about this specific predicted interaction, regardless of how well it scored statistically.


In [ ]:
saveRDS(object = prostate, file = "spatial_prostate_2.rds")

In [ ]:
sessionInfo()